# XGBoost validation with a three-fight minimum

This notebook uses `w_pre_fight_stats.csv`, applies the configured minimum-history filter, and evaluates the requested XGBoost configuration across five stratified 10% splits. Each split is balanced by Red/Blue mirroring. At the end, the same pipeline trains one final model on the full eligible balanced dataset and saves it under `winner/models/`.

In [ ]:
from pathlib import Path

import joblib

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 99
SPLIT_SEEDS = [111, 222, 333, 444, 556]
TEST_SIZE = 0.10
MIN_PRIOR_FIGHTS = 2

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data_clean').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data_clean' / 'w_pre_fight_stats.csv'

MODEL_DIR = PROJECT_ROOT / 'winner' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'w_xgb_min_final.joblib'


XGB_PARAMS = dict(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=-1,
)

DROP_PREFIXES = (
    'B_Stance_', 'R_Stance_', 'B_Stance', 'R_Stance', 'weight_class_'
)
DROP_EXACT = {
    'B_Weight_lbs', 'R_Weight_lbs', 'R_prior_fights', 'B_prior_fights'
}
LEAKAGE_OR_METADATA_COLS = {
    'Winner', 'winner', 'winner_side', 'fight_id', 'date',
    'R_fighter', 'B_fighter', 'win_by', 'fight_type',
    'R_prior_fights', 'B_prior_fights'
}

def keep_feature_col(col):
    if col in DROP_EXACT or col in LEAKAGE_OR_METADATA_COLS:
        return False
    return not col.startswith(DROP_PREFIXES)


## Count prior appearances and filter the dataset

Prior appearances are counted across both Red and Blue sides, using date and fight ID ordering. The minimum-history filter is applied before any train/test split.

In [20]:
def add_prior_fight_counts(data):
    if {'R_prior_fights', 'B_prior_fights'}.issubset(data.columns):
        return data

    base = data[['fight_id', 'date', 'R_fighter', 'B_fighter']].copy()

    red = base.rename(columns={'R_fighter': 'fighter'})[['fight_id', 'date', 'fighter']]
    red['side'] = 'R'
    blue = base.rename(columns={'B_fighter': 'fighter'})[['fight_id', 'date', 'fighter']]
    blue['side'] = 'B'

    appearances = pd.concat([red, blue], ignore_index=True)
    appearances = appearances.sort_values(
        ['fighter', 'date', 'fight_id', 'side']
    ).reset_index(drop=True)
    appearances['prior_fights'] = appearances.groupby('fighter').cumcount()

    red_counts = (
        appearances[appearances['side'] == 'R']
        [['fight_id', 'fighter', 'prior_fights']]
        .rename(columns={'fighter': 'R_fighter', 'prior_fights': 'R_prior_fights'})
    )
    blue_counts = (
        appearances[appearances['side'] == 'B']
        [['fight_id', 'fighter', 'prior_fights']]
        .rename(columns={'fighter': 'B_fighter', 'prior_fights': 'B_prior_fights'})
    )

    result = data.merge(red_counts, on=['fight_id', 'R_fighter'], how='left')
    return result.merge(blue_counts, on=['fight_id', 'B_fighter'], how='left')

data = pd.read_csv(DATA_PATH)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data = add_prior_fight_counts(data)

filtered_data = data[
    data['Winner'].isin(['Red', 'Blue'])
    & data['date'].notna()
    & (data['R_prior_fights'] >= MIN_PRIOR_FIGHTS)
    & (data['B_prior_fights'] >= MIN_PRIOR_FIGHTS)
].copy().reset_index(drop=True)

feature_cols = [col for col in filtered_data.columns if keep_feature_col(col)]

print('Input rows:', len(data))
print('Rows after valid-winner and minimum-history filter:', len(filtered_data))
print('Target distribution:')
print(filtered_data['Winner'].value_counts())
print('Feature columns before Red/Blue differences:', len(feature_cols))


Input rows: 8763
Rows after valid-winner and minimum-history filter: 4915
Target distribution:
Winner
Red     2986
Blue    1929
Name: count, dtype: int64
Feature columns before Red/Blue differences: 133


## Prepare training folds and shared-scaled differences

For each validation seed, the eligible dataset is randomly balanced before splitting by mirroring selected majority-side fights: every paired R_/B_ column is swapped and the Winner label is changed. Imputation and scaling are still fitted separately inside each training split.

In [21]:
def paired_suffixes(columns):
    return sorted({
        col[2:] for col in columns
        if col.startswith('R_') and f'B_{col[2:]}' in columns
    })

def mirror_side_rows(rows, columns):
    mirrored = rows.copy()
    for suffix in paired_suffixes(columns):
        r_col, b_col = f'R_{suffix}', f'B_{suffix}'
        r_values = mirrored[r_col].copy()
        mirrored[r_col] = mirrored[b_col].to_numpy()
        mirrored[b_col] = r_values.to_numpy()
    mirrored['Winner'] = mirrored['Winner'].map({'Red': 'Blue', 'Blue': 'Red'})
    return mirrored

def balance_rows(data, seed):
    rng = np.random.default_rng(seed)
    balanced = data.copy().reset_index(drop=True)
    counts = balanced['Winner'].value_counts()
    if len(counts) < 2 or counts.nunique() == 1:
        return balanced

    majority = counts.idxmax()
    target_majority = int(np.ceil(counts.sum() / 2))
    n_to_flip = int(counts[majority] - target_majority)
    if n_to_flip > 0:
        majority_idx = balanced.index[balanced['Winner'] == majority].to_numpy()
        flip_idx = rng.choice(majority_idx, size=n_to_flip, replace=False)
        flip_rows = balanced.loc[flip_idx]
        balanced = balanced.drop(index=flip_idx)
        balanced = pd.concat([balanced, mirror_side_rows(flip_rows, balanced.columns)], ignore_index=True)
    return balanced.reset_index(drop=True)

def make_y(data):
    return data['Winner'].map({'Blue': 0, 'Red': 1}).astype(int)

def fit_shared_scaled_diff(train_data, test_data, columns, return_preprocessors=False):
    train_out = pd.DataFrame(index=train_data.index)
    test_out = pd.DataFrame(index=test_data.index)
    preprocessors = {'context': None, 'paired': {}}

    context_cols = [col for col in columns if not col.startswith(('R_', 'B_'))]
    if context_cols:
        imputer = SimpleImputer(strategy='median')
        scaler = StandardScaler()
        train_context = train_data[context_cols].apply(pd.to_numeric, errors='coerce')
        test_context = test_data[context_cols].apply(pd.to_numeric, errors='coerce')
        train_imputed = imputer.fit_transform(train_context)
        train_out[context_cols] = scaler.fit_transform(train_imputed)
        test_out[context_cols] = scaler.transform(imputer.transform(test_context))
        preprocessors['context'] = {
            'columns': context_cols,
            'imputer': imputer,
            'scaler': scaler,
        }

    for suffix in paired_suffixes(columns):
        r_col, b_col = f'R_{suffix}', f'B_{suffix}'
        train_r = pd.to_numeric(train_data[r_col], errors='coerce')
        train_b = pd.to_numeric(train_data[b_col], errors='coerce')
        combined = pd.concat([train_r, train_b], ignore_index=True)
        median = combined.median()
        median = 0.0 if pd.isna(median) else median
        filled = combined.fillna(median)
        mean, std = filled.mean(), filled.std(ddof=0)
        std = 1.0 if pd.isna(std) or std == 0 else std

        train_r_scaled = (train_r.fillna(median) - mean) / std
        train_b_scaled = (train_b.fillna(median) - mean) / std
        test_r_scaled = (pd.to_numeric(test_data[r_col], errors='coerce').fillna(median) - mean) / std
        test_b_scaled = (pd.to_numeric(test_data[b_col], errors='coerce').fillna(median) - mean) / std

        train_out[f'DIFF_{suffix}'] = train_r_scaled - train_b_scaled
        test_out[f'DIFF_{suffix}'] = test_r_scaled - test_b_scaled
        preprocessors['paired'][suffix] = {
            'r_column': r_col,
            'b_column': b_col,
            'median': median,
            'mean': mean,
            'std': std,
        }

    if return_preprocessors:
        return train_out, test_out, preprocessors
    return train_out, test_out

def make_xgb_model(seed):
    return XGBClassifier(random_state=seed, **XGB_PARAMS)


## Run five validation splits

Each split creates its own randomly balanced eligible dataset, fits a fresh scaler on the training portion, and evaluates on the untouched validation portion. The final table reports the validation metrics.

In [22]:
results = []
importance_rows = []

for split_num, seed in enumerate(SPLIT_SEEDS, start=1):
    balanced_data = balance_rows(filtered_data, seed)
    y = make_y(balanced_data)
    print(f'Validation seed {seed} target distribution:')
    print(balanced_data['Winner'].value_counts().to_dict())

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    train_idx, test_idx = next(splitter.split(balanced_data, y))

    train_raw = balanced_data.iloc[train_idx].copy()
    test_raw = balanced_data.iloc[test_idx].copy()

    X_train, X_test = fit_shared_scaled_diff(train_raw, test_raw, feature_cols)
    y_train, y_test = make_y(train_raw), make_y(test_raw)

    model = make_xgb_model(seed)
    model.fit(X_train, y_train)
    importance_rows.append(pd.Series(model.feature_importances_, index=X_train.columns, name=seed))
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'split': split_num,
        'seed': seed,
        'train_rows': len(train_raw),
        'test_rows': len(test_raw),
        'accuracy': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

summary = results_df.agg(
    mean_accuracy=('accuracy', 'mean'),
    std_accuracy=('accuracy', 'std'),

)

print('Five-split validation summary:')
display(summary.round(4))


Validation seed 111 target distribution:
{'Red': 2458, 'Blue': 2457}
Validation seed 222 target distribution:
{'Red': 2458, 'Blue': 2457}
Validation seed 333 target distribution:
{'Red': 2458, 'Blue': 2457}
Validation seed 444 target distribution:
{'Red': 2458, 'Blue': 2457}
Validation seed 556 target distribution:
{'Red': 2458, 'Blue': 2457}


,split,seed,train_rows,test_rows,accuracy,balanced_accuracy,roc_auc
0,1,111,4423,492,0.5955,0.5955,0.6383
1,2,222,4423,492,0.6118,0.6118,0.6476
2,3,333,4423,492,0.6240,0.6240,0.6658
3,4,444,4423,492,0.6321,0.6321,0.6923
4,5,556,4423,492,0.6159,0.6159,0.6707


Five-split validation summary:


,accuracy
mean_accuracy,0.6159
std_accuracy,0.0138


## Interpretation note

These are random stratified validation splits, matching the prior XGBoost experiment. A chronological holdout should also be run before treating the result as an estimate of future-fight performance.

## XGBoost feature loadings

XGBoost does not have linear-model coefficients; these are normalized tree feature importances. The table averages each feature's importance across the five fitted validation models and includes its standard deviation.

In [23]:
importance_by_seed = pd.DataFrame(importance_rows)

loadings = pd.DataFrame({
    'feature': importance_by_seed.columns,
    'mean_loading': importance_by_seed.mean(axis=0).to_numpy(),
    'std_loading': importance_by_seed.std(axis=0).to_numpy(),
}).sort_values('mean_loading', ascending=False).reset_index(drop=True)

print('Top 30 average XGBoost feature loadings:')
display(loadings.head(30).round(6))


Top 30 average XGBoost feature loadings:


,feature,mean_loading,std_loading
0,DIFF_age,0.036906,0.001908
1,DIFF_losses,0.029288,0.003820
2,DIFF_longest_win_streak,0.021500,0.002018
3,DIFF_avg_opp_TOTAL_STR_landed,0.021200,0.006133
4,DIFF_wins,0.020649,0.003157
5,DIFF_avg_opp_SIG_STR_pct,0.020492,0.001760
6,DIFF_win_by_Decision_Split,0.020258,0.001729
7,DIFF_avg_GROUND_att,0.020017,0.003719
8,DIFF_avg_opp_SIG_STR_landed,0.020010,0.000856
9,DIFF_avg_opp_TOTAL_STR_att,0.019399,0.004359


## Train and save the final full-dataset model

This trains one final model using every row that passed the minimum-history filter. The full eligible dataset is balanced once using `RANDOM_STATE`, then the model and the fitted imputation/scaling metadata are saved for later prediction.

In [24]:
final_balanced_data = balance_rows(filtered_data, RANDOM_STATE)
X_final, _, final_preprocessors = fit_shared_scaled_diff(
    final_balanced_data,
    final_balanced_data,
    feature_cols,
    return_preprocessors=True,
)
y_final = make_y(final_balanced_data)

final_model = make_xgb_model(RANDOM_STATE)
final_model.fit(X_final, y_final)

joblib.dump({
    'model': final_model,
    'preprocessors': final_preprocessors,
    'raw_feature_columns': feature_cols,
    'model_feature_columns': X_final.columns.tolist(),
    'target_mapping': {'Blue': 0, 'Red': 1},
    'xgb_params': XGB_PARAMS,
    'min_prior_fights': MIN_PRIOR_FIGHTS,
    'balance_seed': RANDOM_STATE,
    'training_rows': len(final_balanced_data),
}, MODEL_PATH)

print(f'Final training rows: {len(final_balanced_data)}')
print('Final target distribution:')
print(final_balanced_data['Winner'].value_counts())
print(f'Saved final model: {MODEL_PATH}')


Final training rows: 4915
Final target distribution:
Winner
Red     2458
Blue    2457
Name: count, dtype: int64
Saved final model: e:\Libraries\Documents\GitHub\UFC-Predictions\winner\models\w_xgb_min_final.joblib
